# E1 · 기존 NF4 어댑터를 bf16 베이스에 얹어 다시 채점 (학습 없음)

**하는 일**
- `SRC_RUN`(기존 실행)의 어댑터를 **merge 없이** bf16 베이스 모델에 붙여 검증 1,000문항을 채점합니다. 학습은 하지 않습니다.
- 기존 NF4 결과(`preds_final_ep*__up2.0__1000.csv`)와 **문항 단위로 짝지어** 고침·새로 틀림·NLL을 비교하고, 보고서의 사전 기준(순이득 ≥ +3, NLL 감소)으로 판정합니다. 현재 최고 조합에서 이 멤버만 바꿨을 때의 변화도 함께 보여 줍니다.
- 판정을 통과하면 이어서 test 6,714문항을 채점합니다. 파일 이름은 stage2와 같은 규칙이라, Stage 3에서 멤버 이름만 `…_bf16eval`로 바꾸면 됩니다.

**안전장치**
- 결과는 새 폴더 `vqa_runs/<SRC_RUN>_bf16eval/`에만 씁니다. 기존 실행 폴더는 읽기만 합니다 (공용 로그 `results_log_s2.csv`에는 한 줄씩 추가).
- 본 채점 전에 확인합니다: 어댑터 파일과 모델 가중치가 값까지 같은지, 어댑터를 켜고 끄면 출력이 달라지는지, 이미지 토큰 수가 기존 결과와 같은지(같은 해상도).

**런타임**: G4(96GB) 또는 A100 80GB. bf16 27B는 약 56GB라 A100 40GB에는 올라가지 않습니다(7번 셀에서 멈춤).
**시간(G4 기준 추정)**: 모델 내려받기·로드 약 10분 + 검증 15~20분 → 판정까지 30분 안팎. test까지 채점하면 1~1.5시간이 더 듭니다(진행 중 남은 시간 표시).

**실행**: 1번 셀의 `SRC_RUN`, `MODEL_ID`만 확인하고 위에서부터 끝까지 실행하세요.

## 1. 설정

In [ ]:
SRC_RUN   = "q35_27b_r16_native"      # 다시 채점할 기존 실행 (vqa_runs 아래 폴더 이름). 이 폴더는 읽기만 합니다
MODEL_ID  = "Qwen/Qwen3.5-27B"        # 그 실행의 베이스 모델. 어댑터에 기록된 이름과 다르면 3번 셀에서 멈춥니다
PRESET    = "up2.0"                   # 채점 해상도. 기존 NF4 결과(preds_final_ep*__up2.0__1000.csv)와 같은 해상도로 비교합니다
SRC_EPOCH = "auto"                    # "auto": 기존 최종 평가에 쓰인 epoch의 어댑터 / 숫자를 넣으면 adapter_ep<숫자>
RUN_TEST  = "auto"                    # "auto": 판정을 통과할 때만 test 6,714문항 채점 / True: 항상 / False: 안 함
RUN_NAME  = f"{SRC_RUN}_bf16eval"     # 결과 폴더 이름 = Stage 3에서 쓸 멤버 이름
# 다른 멤버도 같은 방법으로 재채점: SRC_RUN·MODEL_ID 두 줄만 바꾸세요
#   예) "q38_27b_r16_native" + "Qwen/Qwen3.8-27B" / "gemma4_31b_r16_up20" + "google/gemma-4-31B-it"
#   HCX는 별도 노트북(개발 버전 transformers)으로 학습했으므로 이 노트북으로는 안 됩니다

# 판정 기준 (보고서 E1): 같은 1,000문항에서 순이득(고침 − 새로 틀림) ≥ +3 이고 NLL 감소
NET_MIN = 3
# 앙상블 관점(참고): 아래 조합에서 SRC_RUN 자리만 bf16 결과로 바꿨을 때를 같은 1,000문항에서 비교합니다. 건너뛰려면 None
# 인수인계 문서의 clean4_qg_hcx(검증 97.7)를 넣어 두었습니다. 지금 최고 조합(공개 0.97676)이 이것과 다르면 바꾸세요
ENSEMBLE_REF = {"name": "clean4_qg_hcx", "expected_val_acc": 97.7, "weights": [1, 1, 1, 1, 2],
                "members": [("q35_27b_r16_native", "up2.0"), ("q38_27b_r16_native", "up2.0"), ("gemma4_31b_r16_up20_dev", "up2.0"),
                            ("gemma4_31b_r16_up20", "up2.0"), ("hcx32b_r16_native", "up2.0")]}

DRIVE_ZIP    = "/content/drive/MyDrive/ssafy-16-2-ai.zip"
ZIP_PASSWORD = "YOUR_ZIP_PASSWORD"
DATA_DIR     = "/content"
OUT_DIR      = "/content/drive/MyDrive/vqa_runs"
BATCH_SIZE   = "auto"          # 평가용 배치 (OOM이면 자동으로 절반)
SEED, VAL_SIZE = 42, 1000      # 검증 분할용 — 팀 공통값이므로 절대 바꾸지 마세요
SKIP_MISSING_IMAGES = False
HF_TOKEN     = ""              # Gemma처럼 라이선스 승인이 필요한 모델일 때 HF 토큰 (비워 두면 Colab 보안 비밀 'HF_TOKEN'을 찾아봄)

RES_PRESETS = {
    "baseline384": (384 * 384, 384 * 384),
    "mid":         (256 * 28 * 28, 512 * 28 * 28),
    "native":      (256 * 28 * 28, 1280 * 28 * 28),
    "up1.5":       (1_550_000, 2_100_000),
    "up2.0":       (2_700_000, 3_400_000),
}
RES_PRESETS_GEMMA4 = {"baseline384": 70, "mid": 140, "native": 280, "up1.5": 560, "up2.0": 1120}
RES_PRESET = PRESET
import os; os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # GPU 메모리 단편화 완화 (torch import 전에 설정)

## 2. 라이브러리 설치 (학습 때와 같은 버전)

In [ ]:
import re
!pip -q install "transformers==5.17.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" "peft==0.21.0"
# Colab에 기본 설치된 구버전 torchao가 있으면 peft 0.21이 LoRA를 붙이는 순간 ImportError를 냅니다. 이 프로젝트에서는 쓰지 않으므로 제거
!pip -q uninstall -y torchao
# Qwen3.5의 선형 어텐션용 고속 Triton 커널. 없으면 순수 PyTorch 대체 구현으로 돌아 학습 시 메모리 부족(OOM)이 나고 매우 느립니다.
# (transformers가 import 시점에 이 패키지를 찾으므로, 이미 모델을 로드한 세션에서 설치했다면 '세션 다시 시작'이 필요합니다)
if re.search(r"Qwen3\.[5-9]", MODEL_ID):      # Qwen3.5 / 3.6 / 3.8 — 모두 같은 선형 어텐션 구조(model_type qwen3_5)
    !pip -q install "flash-linear-attention==0.5.2"

## 3. Drive 연결 → 원본 실행 확인 → 데이터 준비
원본 실행 폴더·어댑터·기존 결과 파일을 먼저 확인하고, 이름이 틀렸으면 압축을 풀기 전에 멈춥니다.

In [ ]:
import os, re, json, hashlib, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── 원본 실행 확인 (읽기만 합니다). 데이터 압축을 풀기 전에 확인해서, 이름이 틀렸으면 바로 멈춥니다 ──
SRC_DIR = os.path.join(OUT_DIR, SRC_RUN)
assert os.path.isdir(SRC_DIR), f"원본 실행 폴더가 없습니다: {SRC_DIR} → 1번 셀의 SRC_RUN을 확인하세요"
_files = sorted(os.listdir(SRC_DIR))
_rx_val = re.compile(rf"^preds_final_ep(\d+)__{re.escape(PRESET)}__1000\.csv$")
REF_VAL_BY_EP = {int(m.group(1)): os.path.join(SRC_DIR, f) for f in _files if (m := _rx_val.match(f))}
_adapters = sorted(int(m.group(1)) for f in _files if (m := re.match(r"^adapter_ep(\d+)$", f)))
_log_path = os.path.join(SRC_DIR, "train_log.json")
SRC_LOG = json.load(open(_log_path, encoding="utf-8")) if os.path.exists(_log_path) else {}
SRC_CFG = SRC_LOG.get("config") or {}
_ea = {int(k): v for k, v in (SRC_LOG.get("epoch_acc") or {}).items()}
if SRC_EPOCH != "auto":       SRC_EP = int(SRC_EPOCH)
elif len(REF_VAL_BY_EP) == 1: SRC_EP = next(iter(REF_VAL_BY_EP))          # 기존 최종 평가 파일 이름에 적힌 epoch
elif _ea:                     SRC_EP = max(_ea, key=lambda e: (_ea[e], e))  # stage2 11번 셀과 같은 규칙 (동률이면 뒤 epoch)
elif _adapters:               SRC_EP = _adapters[-1]
else: raise FileNotFoundError(f"{SRC_DIR}에 adapter_ep* 폴더가 없습니다. 폴더 내용: {_files}")
ADAPTER_DIR = os.path.join(SRC_DIR, f"adapter_ep{SRC_EP}")
_st = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
assert os.path.isfile(_st) and os.path.isfile(os.path.join(ADAPTER_DIR, "adapter_config.json")), \
    f"어댑터 파일이 없습니다: {ADAPTER_DIR} (원본 폴더 내용: {_files})"
REF_VAL = REF_VAL_BY_EP.get(SRC_EP)
_rx_test = re.compile(rf"^preds_final_ep{SRC_EP}_test__{re.escape(PRESET)}__\d+\.csv$")
REF_TEST = next((os.path.join(SRC_DIR, f) for f in _files if _rx_test.match(f)), None)

ADAPTER_CFG = json.load(open(os.path.join(ADAPTER_DIR, "adapter_config.json"), encoding="utf-8"))
_rec = str(ADAPTER_CFG.get("base_model_name_or_path") or "")
if _rec and not _rec.startswith("/") and _rec != MODEL_ID:
    raise ValueError(f"이 어댑터는 '{_rec}' 위에서 학습됐는데 MODEL_ID는 '{MODEL_ID}'입니다 → 1번 셀의 MODEL_ID를 '{_rec}'로 바꾸세요")
SRC_TRAIN_ON_FULL = bool(SRC_CFG.get("TRAIN_ON_FULL", False))
DO_VAL = not SRC_TRAIN_ON_FULL        # 검증셋까지 학습한 실행이면 검증 비교는 의미가 없음 → test만 채점

def _md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
ADAPTER_MD5 = _md5(_st)
RUN_DIR = os.path.join(OUT_DIR, RUN_NAME)
assert os.path.abspath(RUN_DIR) != os.path.abspath(SRC_DIR), "RUN_NAME이 원본 실행과 같습니다"
os.makedirs(RUN_DIR, exist_ok=True)
INFO_PATH = os.path.join(RUN_DIR, "e1_info.json")
E1_INFO = json.load(open(INFO_PATH, encoding="utf-8")) if os.path.exists(INFO_PATH) else {}
if E1_INFO and E1_INFO.get("adapter_md5") != ADAPTER_MD5:
    raise RuntimeError(f"{RUN_DIR}에는 다른 어댑터로 만든 결과가 있습니다 (기록 {str(E1_INFO.get('adapter_md5'))[:8]} ≠ 지금 {ADAPTER_MD5[:8]}) "
                       "→ 1번 셀의 RUN_NAME을 바꾸세요")
E1_INFO.update({"src_run": SRC_RUN, "adapter_dir": ADAPTER_DIR, "adapter_md5": ADAPTER_MD5, "model_id": MODEL_ID,
                "precision": "bf16 베이스 + 기존 어댑터 (merge 없음)"})
json.dump(E1_INFO, open(INFO_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

_nf4_rule = re.search(r"(?<![\d.])(2\d|3\d|7\d)B", _rec or MODEL_ID) is not None       # 기존 노트북의 USE_4BIT="auto" 규칙
print(f"원본 실행: {SRC_DIR}")
print(f"  어댑터 adapter_ep{SRC_EP} (md5 {ADAPTER_MD5[:8]}) | r={ADAPTER_CFG.get('r')}, alpha={ADAPTER_CFG.get('lora_alpha')} | "
      f"학습 해상도 {SRC_CFG.get('TRAIN_PRESET', '?')} | 기록된 베이스 {_rec or '?'}")
print(f"  비교 기준(기존) 검증 파일: {os.path.basename(REF_VAL) if REF_VAL else '없음 → 비교 불가'} | "
      f"기존 test 파일: {os.path.basename(REF_TEST) if REF_TEST else '없음'}")
print("  기존 노트북의 USE_4BIT='auto' 규칙이었다면 이 모델은 " +
      ("NF4로 학습·평가됐습니다 (이번 비교 대상)" if _nf4_rule else "bf16이었습니다 → 재채점해도 거의 달라지지 않을 것입니다"))
if SRC_TRAIN_ON_FULL: print("  ⚠️ 검증셋까지 학습한 실행입니다 → 검증 비교는 건너뛰고 test만 채점합니다")
print(f"결과 폴더(새로 씀): {RUN_DIR}")

# ── 데이터 (stage2와 동일) ──
if IN_COLAB and not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
    print("압축 해제 중... (몇 분 걸립니다)")
    r = subprocess.run(["7z", "x", "-y", f"-p{ZIP_PASSWORD}", f"-o{DATA_DIR}", DRIVE_ZIP], capture_output=True, text=True)
    print(r.stdout[-400:]); assert r.returncode == 0, r.stderr[-800:]
if not HF_TOKEN and IN_COLAB:
    try:
        from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception: pass
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN, add_to_git_credential=False); print("Hugging Face 로그인 완료")
for f in ["train.csv", "test.csv", "sample_submission.csv"]:
    assert os.path.exists(os.path.join(DATA_DIR, f)), f"{f} 가 {DATA_DIR} 에 없습니다. DATA_DIR / 압축 비밀번호를 확인하세요."
print("데이터 OK:", DATA_DIR)

## 4. 공통 import · CSV 로드

In [ ]:
import re, json, time, math, hashlib, random, datetime, platform
import numpy as np, pandas as pd, torch
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def _fix_nvrtc():
    # 일부 Colab 이미지(새 CUDA)에서 GPU 정수 텐서의 prod가 런타임 컴파일(NVRTC) 도중 내장 라이브러리를 못 찾아 실패함.
    # 1) 그 라이브러리를 경로로 직접 불러오고, 2) 그래도 안 되면 정수 prod만 CPU에서 계산하도록 우회 (결과는 완전히 같음)
    if not torch.cuda.is_available(): return
    import glob, ctypes
    def _ok():
        try: torch.tensor([[1, 2, 3]], device="cuda").prod(-1).item(); return True, ""
        except RuntimeError as e: return False, str(e)
    good, msg = _ok()
    if good: return
    need = re.findall(r"libnvrtc-builtins\.so[\d.]*", msg)
    pats = ["/usr/local/lib/python3*/dist-packages/nvidia/**/libnvrtc-builtins.so*", "/usr/local/lib/python3*/site-packages/nvidia/**/libnvrtc-builtins.so*",
            "/usr/local/cuda*/lib64/libnvrtc-builtins.so*", "/usr/lib/x86_64-linux-gnu/libnvrtc-builtins.so*"]
    found = sorted({f for p in pats for f in glob.glob(p, recursive=True)})
    for f in found:
        if not need or any(os.path.basename(f).startswith(n) for n in need):
            try: ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
            except OSError: pass
    if _ok()[0]: print("✅ NVRTC 내장 라이브러리를 직접 불러와 해결:", need); return
    _orig_prod = torch.Tensor.prod
    def _prod(self, *a, **k):
        if self.is_cuda and not (self.is_floating_point() or self.is_complex()):
            return _orig_prod(self.cpu(), *a, **k).to(self.device)
        return _orig_prod(self, *a, **k)
    torch.Tensor.prod = _prod
    print("⚠️ GPU 정수 prod를 CPU에서 계산하도록 우회 (결과 동일) | 확인:", _ok()[0], "| 찾은 라이브러리:", found or "없음")
_fix_nvrtc()

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sub_df   = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
dev_df   = pd.read_csv(os.path.join(DATA_DIR, "dev.csv")) if os.path.exists(os.path.join(DATA_DIR, "dev.csv")) else None
for c in ["question", "a", "b", "c", "d"]:
    train_df[c] = train_df[c].astype(str); test_df[c] = test_df[c].astype(str)
train_df["answer"] = train_df["answer"].astype(str).str.strip().str.lower()
assert set(train_df.answer.unique()) <= set("abcd"), train_df.answer.unique()
assert list(test_df.id) == list(sub_df.id), "test.csv 와 sample_submission.csv 의 id 순서가 다릅니다"
print(f"train {len(train_df):,} | test {len(test_df):,} | dev {0 if dev_df is None else len(dev_df):,}")
print("정답 분포:", train_df.answer.value_counts().sort_index().to_dict())

def img_path(rel):            # CSV의 path 컬럼은 DATA_DIR 기준 상대경로
    return os.path.join(DATA_DIR, rel)

## 5. 고정 검증셋 분할 (기존 결과와 같아야 비교할 수 있음)

In [ ]:
EXPECTED_SPLIT_MD5 = "df05e4bc19d27ebceeef2570672149c2"

def make_split(df, val_size=VAL_SIZE, seed=SEED):
    rng = np.random.RandomState(seed)
    val_idx = []
    for letter in "abcd":
        idx = np.where(df.answer.values == letter)[0]
        rng.shuffle(idx)
        val_idx.extend(idx[: val_size // 4].tolist())
    val_idx = sorted(val_idx)
    mask = np.zeros(len(df), dtype=bool); mask[val_idx] = True
    return df[~mask].reset_index(drop=True), df[mask].reset_index(drop=True)

tr_df, val_df = make_split(train_df)
split_md5 = hashlib.md5(",".join(val_df.id.tolist()).encode()).hexdigest()
print(f"train {len(tr_df):,} / valid {len(val_df):,} | split md5 = {split_md5}")
print("valid 정답 분포:", val_df.answer.value_counts().sort_index().to_dict())
assert split_md5 == EXPECTED_SPLIT_MD5, "⚠️ 검증 분할이 기준과 다릅니다 → 기존 결과와 비교할 수 없습니다. train.csv 원본/SEED/VAL_SIZE를 확인하세요."
print("✅ 팀 공통 분할과 일치")

## 6. 프롬프트 (학습 때와 동일)

In [ ]:
PROMPT_VERSION = "baseline_v1"
SYSTEM_INSTRUCT = ("You are a helpful visual question answering assistant. "
                   "Answer using exactly one letter among a, b, c, or d. No explanation.")
LETTERS = "abcd"

def build_mc_prompt(question, options):
    a, b, c, d = options
    return (f"{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")

def build_messages(row, perm=(0, 1, 2, 3)):
    # perm[k] = 화면의 k번째 자리에 놓을 '원래 선지'의 인덱스
    opts = [str(row[LETTERS[j]]) for j in perm]
    user_text = build_mc_prompt(str(row["question"]), opts)
    if globals().get("USE_SYSTEM_ROLE", True):
        return [{"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]
    # system 역할이 없는 템플릿(일부 Gemma 등): 지시문을 user 메시지 앞에 붙임
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": SYSTEM_INSTRUCT + "\n\n" + user_text}]}]

def load_image(rel):
    im = Image.open(img_path(rel))
    im = ImageOps.exif_transpose(im)      # 촬영 방향(EXIF) 보정 — 없으면 그대로
    return im.convert("RGB")

## 7. bf16 베이스 로드 + 기존 어댑터 부착 + 적용 확인
양자화 없이 bf16으로 올린 뒤 어댑터를 merge 없이 얹습니다. 어댑터 파일의 LoRA 텐서가 전부 값 그대로 들어갔는지 확인합니다.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel, load_peft_weights, get_peft_model_state_dict
import transformers, peft, gc
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)

for _v in ("model", "base", "enc", "logits", "out"):       # 같은 세션에서 다시 실행할 때 이전 모델을 GPU에서 비움
    if _v in globals(): del globals()[_v]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

assert torch.cuda.is_available(), "GPU 런타임이 아닙니다 (런타임 → 런타임 유형 변경)"
DEVICE = "cuda"
GPU_NAME = torch.cuda.get_device_name(0); cap = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
assert cap[0] >= 8, f"{GPU_NAME}은 bf16 연산을 지원하지 않습니다 → G4 또는 A100 80GB 런타임을 쓰세요"
DTYPE = torch.bfloat16
_sz = re.search(r"(?<![\d.])(\d+)B", MODEL_ID, flags=re.I)
NEED_GB = int(_sz.group(1)) * 2.05 + 6 if _sz else 0.0       # bf16 가중치(파라미터 x 2바이트) + 채점용 여유
if VRAM_GB < NEED_GB:
    raise RuntimeError(f"bf16 {MODEL_ID}에는 GPU 메모리가 약 {NEED_GB:.0f}GB 필요한데 {GPU_NAME}은 {VRAM_GB:.0f}GB입니다 "
                       "→ 런타임을 G4(96GB) 또는 A100 80GB로 바꾸세요")
BS = (8 if VRAM_GB >= 38 else 4 if VRAM_GB >= 20 else 2) if BATCH_SIZE == "auto" else int(BATCH_SIZE)
print(f"GPU: {GPU_NAME} ({VRAM_GB:.0f}GB) | dtype={DTYPE} | 4bit=False (bf16 고정) | 평가 배치={BS}")

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"
tok = processor.tokenizer
FAMILY = "gemma4" if type(processor).__name__.startswith("Gemma4") else "qwen"     # 프로세서 계열 (해상도 인자·이미지 전달 방식이 다름)
IMAGE_TOKEN_ID = getattr(processor, "image_token_id", None)
if IMAGE_TOKEN_ID is None: IMAGE_TOKEN_ID = tok.convert_tokens_to_ids(getattr(processor, "image_token", "<|image_pad|>"))
_tmpl = processor.chat_template if isinstance(processor.chat_template, str) else json.dumps(processor.chat_template)
TEMPLATE_KW = {"enable_thinking": False} if "enable_thinking" in (_tmpl or "") else {}
try:
    processor.apply_chat_template(build_messages({"question": "q", "a": "1", "b": "2", "c": "3", "d": "4"}), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW); USE_SYSTEM_ROLE = True
except Exception as _e:
    USE_SYSTEM_ROLE = False; print("system 역할 미지원 템플릿 → 지시문을 user 메시지 앞에 붙입니다")
print(f"계열: {FAMILY} | 이미지 토큰 id: {IMAGE_TOKEN_ID} | chat template 추가 인자: {TEMPLATE_KW} | system 역할: {USE_SYSTEM_ROLE}")

t0 = time.time()
base = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=DTYPE, attn_implementation="sdpa", device_map={"": 0})
base.config.use_cache = False
_quant = [n for n, m in base.named_modules() if any(k in type(m).__name__ for k in ("Linear4bit", "Linear8bit"))]
assert not _quant, f"양자화된 층이 있습니다 (bf16이어야 함): {_quant[:3]}"
model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=False)      # merge 하지 않고 어댑터를 그대로 얹음
model.eval()
_dt = sorted({str(p.dtype) for n, p in base.named_parameters() if "lora_" not in n})
_mem = torch.cuda.memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
print(f"모델 로드 {time.time() - t0:.0f}s | {type(base).__name__} | 베이스 가중치 dtype {_dt} | 사용 VRAM {_mem:.1f}GB")
if "qwen3_5" in str(getattr(base.config, "model_type", "")):
    import importlib.util
    if importlib.util.find_spec("fla") is None:
        print("⚠️⚠️ flash-linear-attention이 없습니다 → 선형 어텐션이 순수 PyTorch로 돌아 매우 느립니다. 2번 셀을 실행하고 '세션 다시 시작' 후 처음부터 실행하세요.")
    else:
        print("✅ flash-linear-attention 커널 사용")

n_lora = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A"))
n_lora_vis = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A") and any(k in n for k in ("visual", "vision_tower", "vision_model")))
print(f"LoRA가 붙은 층: {n_lora}개 (비전 타워 {n_lora_vis}개 — 0이어야 정상)")
assert n_lora > 0 and n_lora_vis == 0

def _norm_key(k):
    k = k.replace("base_model.model.", "", 1)
    i = k.rfind("language_model.")
    return k[i + len("language_model."):] if i >= 0 else k

def check_adapter_weights(model, adapter_dir):
    # 어댑터 파일의 LoRA 텐서가 전부, 값 그대로 모델에 들어갔는지 확인 (PEFT는 이름이 안 맞는 키를 조용히 건너뛸 수 있음)
    fw = load_peft_weights(adapter_dir, device="cpu")
    mw = {k: v.detach() for k, v in get_peft_model_state_dict(model).items()}
    same_names = set(fw) == set(mw)
    if not same_names:                                     # 이름 형식(접두어)만 다른 경우를 대비해 언어 모델 기준 이름으로 맞춰 봄
        fw = {_norm_key(k): v for k, v in fw.items()}; mw = {_norm_key(k): v for k, v in mw.items()}
    only_f, only_m = sorted(set(fw) - set(mw)), sorted(set(mw) - set(fw))
    assert not only_f and not only_m, (f"어댑터 파일과 모델의 LoRA 키가 다릅니다 → 파일에만 {len(only_f)}개(예: {only_f[:2]}), "
                                       f"모델에만 {len(only_m)}개(예: {only_m[:2]})")
    worst, nz_b = 0.0, 0
    for k, v in fw.items():
        w = mw[k]
        assert tuple(v.shape) == tuple(w.shape), f"모양이 다릅니다: {k} {tuple(v.shape)} vs {tuple(w.shape)}"
        worst = max(worst, (v.to(w.dtype).float() - w.float().cpu()).abs().max().item())   # 불러올 때와 같은 dtype으로 맞추면 정확히 같아야 함
        if "lora_B" in k and v.abs().max().item() > 0: nz_b += 1
    n_b = sum("lora_B" in k for k in fw)
    assert worst == 0.0, f"어댑터 파일과 모델의 LoRA 값이 다릅니다 (최대 차이 {worst:.3g})"
    assert nz_b > 0, "어댑터 파일의 lora_B가 전부 0입니다 → 학습되지 않은 어댑터입니다"
    return {"keys": len(fw), "same_names": same_names, "nonzero_B": f"{nz_b}/{n_b}", "max_diff": worst}

_c = check_adapter_weights(model, ADAPTER_DIR)
print(f"✅ 어댑터 파일 ↔ 모델 가중치 일치: 키 {_c['keys']}개 전부 값까지 같음 | 0이 아닌 lora_B {_c['nonzero_B']}"
      + ("" if _c["same_names"] else " (이름 접두어만 달라 언어 모델 기준으로 맞춤)"))

LETTER_IDS = []                                     # 첫 번째(소문자·공백 없음)가 학습 목표, 나머지 변형은 확률 집계용
for L in LETTERS:
    ids = []
    for s in (L, L.upper(), " " + L, " " + L.upper()):
        t = tok.encode(s, add_special_tokens=False)
        if len(t) == 1 and t[0] not in ids: ids.append(t[0])
    assert ids and len(tok.encode(L, add_special_tokens=False)) == 1, f"'{L}' 가 단일 토큰이 아닙니다"
    LETTER_IDS.append(ids)
print("글자 토큰 id:", dict(zip(LETTERS, LETTER_IDS)))

## 8. 채점 함수 (학습 때와 동일한 로짓 비교)
stage2의 채점 코드를 그대로 씁니다. 학습용 이미지 증강 부분만 뺐습니다.

In [ ]:
IDENTITY = (0, 1, 2, 3)
TTA_PERMS = [(0, 1, 2, 3), (1, 2, 3, 0), (2, 3, 0, 1), (3, 0, 1, 2)]

def set_resolution(preset):
    global RES_PRESET, PIXEL_KW
    RES_PRESET = preset
    if FAMILY == "gemma4":
        PIXEL_KW = {"max_soft_tokens": int(RES_PRESETS_GEMMA4[preset])}
    else:
        mn, mx = RES_PRESETS[preset]
        PIXEL_KW = {"min_pixels": int(mn), "max_pixels": int(mx)}
set_resolution(PRESET)

def render(row, perm=IDENTITY):
    return processor.apply_chat_template(build_messages(row, perm), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW)

def encode_batch(rows, perms):
    texts  = [render(r, p) for r, p in zip(rows, perms)]
    images = [load_image(r["path"]) for r in rows]
    if FAMILY == "gemma4": images = [[im] for im in images]          # Gemma 프로세서는 샘플별 이미지 리스트를 받음
    return processor(text=texts, images=images, padding=True, return_tensors="pt", **PIXEL_KW)

def _last_logits(enc):
    enc = enc.to(model.device)
    with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
        try:    out = model(**enc, logits_to_keep=1, use_cache=False)
        except TypeError: out = model(**enc)
    return out.logits[:, -1, :].float()

def unpermute(prob_slot, perms):
    probs = np.zeros_like(prob_slot)
    for i, p in enumerate(perms):
        for k, j in enumerate(p): probs[i, j] = prob_slot[i, k]
    return probs

@torch.inference_mode()
def score_rows(rows, perms=None):
    global BS
    perms = perms or [IDENTITY] * len(rows)
    enc = logits = None; oom = False
    try:
        enc = encode_batch(rows, perms)
        n_img_tok = (enc["input_ids"] == IMAGE_TOKEN_ID).sum(dim=1).tolist()
        logits = _last_logits(enc)
        if not torch.isfinite(logits).all(): raise FloatingPointError("로짓에 NaN/Inf 발생")
    except torch.cuda.OutOfMemoryError:
        oom = True
    if oom:
        del enc, logits; torch.cuda.empty_cache()
        if len(rows) == 1: raise RuntimeError("배치 1에서도 GPU 메모리 부족")
        h = len(rows) // 2; BS = max(1, min(BS, h)); print(f"  ⚠️ OOM → 평가 배치를 {BS}(으)로 줄입니다")
        p1, m1, t1 = score_rows(rows[:h], perms[:h]); p2, m2, t2 = score_rows(rows[h:], perms[h:])
        return np.concatenate([p1, p2]), np.concatenate([m1, m2]), t1 + t2
    logp = torch.log_softmax(logits, dim=-1)
    slot = torch.stack([torch.logsumexp(logp[:, ids], dim=1) for ids in LETTER_IDS], dim=1)
    mass = slot.exp().sum(dim=1).cpu().numpy()
    return unpermute(torch.softmax(slot, dim=1).cpu().numpy().astype(np.float64), perms), mass, n_img_tok

def score_rows_tta(rows, tta):
    ps, ms, nt = [], [], None
    for perm in (TTA_PERMS if tta else [IDENTITY]):
        p, m, nt = score_rows(rows, [perm] * len(rows)); ps.append(p); ms.append(m)
    return np.mean(ps, axis=0), np.mean(ms, axis=0), nt

def predict_df(df, tag, tta=False, resume=True, verbose=True, save_every=25):
    # df 전체를 채점. resume=True면 CSV에 이어 쓰며, 같은 tag로 다시 부르면 남은 문항만 처리
    was_training = model.training; model.eval()
    out_path = os.path.join(RUN_DIR, f"preds_{tag}__{RES_PRESET}{'__tta4' if tta else ''}__{len(df)}.csv")
    done_ids = set(pd.read_csv(out_path).id) if (resume and os.path.exists(out_path)) else set()
    if not resume and os.path.exists(out_path): os.remove(out_path)
    todo = df[~df.id.isin(done_ids)]
    if SKIP_MISSING_IMAGES: todo = todo[todo.path.map(lambda p: os.path.exists(img_path(p)))]
    rows = todo.to_dict("records"); recs, t0, n_done, nb = [], time.time(), 0, 0
    if verbose: print(f"[{os.path.basename(out_path)}] 전체 {len(df):,} | 완료 {len(done_ids):,} | 남음 {len(rows):,}")
    while n_done < len(rows):
        batch = rows[n_done: n_done + BS]
        probs, mass, ntok = score_rows_tta(batch, tta)
        for r, p, m, t in zip(batch, probs, mass, ntok):
            recs.append({"id": r["id"], "pred": LETTERS[int(p.argmax())], "p_a": p[0], "p_b": p[1], "p_c": p[2], "p_d": p[3],
                         "letter_mass": float(m), "img_tokens": int(t)})
        n_done += len(batch); nb += 1
        if nb % save_every == 0 or n_done >= len(rows):
            pd.DataFrame(recs).to_csv(out_path, mode="a", header=not os.path.exists(out_path), index=False); recs = []
            if verbose:
                el = time.time() - t0
                print(f"  {n_done:>5}/{len(rows)} | {el / n_done:.2f}s/문항 | 남은 시간 ~{(len(rows) - n_done) * el / n_done / 60:.0f}분", flush=True)
    if was_training: model.train()
    return pd.read_csv(out_path).drop_duplicates("id", keep="last"), out_path

CATS = {"전화번호": r"전화|번호|연락처", "가격/금액": r"가격|얼마|금액|요금|할인", "이름/상호/제목": r"이름|상호|제목|브랜드|가게|매장|간판|업체",
        "날짜/시간": r"날짜|언제|시간|기간|요일|몇 시|년도|연도", "위치/방향/층": r"어디|위치|방향|몇 층|층에|왼쪽|오른쪽|위쪽|아래",
        "개수/숫자": r"몇 개|몇 명|몇 번|개수|몇 가지|숫자", "색상": r"색|컬러", "부정형": r"아닌|않는|없는|틀린|잘못"}

def evaluate(df, tag, tta=False, resume=True, verbose=True, by_type=True):
    pred, _ = predict_df(df, tag, tta=tta, resume=resume, verbose=verbose)
    m = df.merge(pred, on="id", how="inner"); m["correct"] = (m.pred == m.answer)
    acc = float(m.correct.mean()); n = len(m); se = math.sqrt(max(acc * (1 - acc), 1e-9) / n)
    print(f"▶ {tag} | {RES_PRESET}{' +TTA4' if tta else ''} | 정확도 {acc * 100:.2f}% (n={n}, ±{1.96 * se * 100:.1f}%p) | letter_mass {m.letter_mass.mean():.3f}")
    if by_type:
        for k, pat in CATS.items():
            sel = m.question.str.contains(pat)
            if sel.sum() >= 10: print(f"    {k:<12} {m.correct[sel].mean() * 100:5.1f}%  (n={int(sel.sum())})")
    lp = os.path.join(OUT_DIR, "results_log_s2.csv")
    log = {"time": datetime.datetime.now().strftime("%m-%d %H:%M"), "run": f"{RUN_NAME}/{tag}", "model": str(MODEL_ID),
           "res": RES_PRESET, "tta": int(tta), "n": n, "acc": round(acc * 100, 2), "ci95": round(1.96 * se * 100, 2),
           "letter_mass": round(float(m.letter_mass.mean()), 3), "gpu": GPU_NAME, "split_md5": split_md5[:8]}
    pd.DataFrame([log]).to_csv(lp, mode="a", header=not os.path.exists(lp), index=False)
    return acc, m

## 9. 사전 점검 (8문항)
어댑터를 켜고 끈 출력 차이, letter_mass, 기존 결과와의 이미지 토큰 수를 확인합니다. 하나라도 이상하면 여기서 멈춥니다.

In [ ]:
set_resolution(PRESET)
_chk = val_df.head(8)
_rows = _chk.to_dict("records")
t0 = time.time()
p_on, m_on, nt_on = score_rows(_rows)
with model.disable_adapter():
    p_off, m_off, _ = score_rows(_rows)
d_onoff = float(np.abs(p_on - p_off).max())
print(f"8문항 채점 {time.time() - t0:.0f}s | 어댑터 켬/끔 확률 최대 차이 {d_onoff:.3f} | letter_mass 평균 {m_on.mean():.3f} | 이미지 토큰 {nt_on}")
assert d_onoff > 1e-3, "어댑터를 켜고 꺼도 출력이 같습니다 → 어댑터가 적용되지 않았습니다. 위 출력을 확인하세요"
assert m_on.mean() > 0.9, "letter_mass가 낮습니다 → 프롬프트·토큰 처리 문제입니다. 위 출력을 확인하세요"
if REF_VAL:
    _ref = pd.read_csv(REF_VAL).drop_duplicates("id", keep="last").set_index("id").reindex(_chk.id)
    if _ref.pred.isna().any():
        print("⚠️ 기존 검증 파일에 이 8문항이 없습니다 → 사전 비교는 건너뜁니다")
    else:
        _pref = _ref[["p_a", "p_b", "p_c", "p_d"]].to_numpy(dtype=float)
        _agree = int((p_on.argmax(1) == _pref.argmax(1)).sum())
        print(f"기존 결과와 비교(같은 8문항): 답 일치 {_agree}/8 | 확률 평균 차이 {np.abs(p_on - _pref).mean():.4f} | "
              f"letter_mass {_ref.letter_mass.mean():.3f} → {m_on.mean():.3f}")
        if "img_tokens" in _ref.columns:
            assert [int(t) for t in _ref.img_tokens] == [int(t) for t in nt_on], \
                f"이미지 토큰 수가 기존 결과와 다릅니다 ({[int(t) for t in _ref.img_tokens]} vs {list(nt_on)}) → 해상도·프로세서가 달라 비교할 수 없습니다"
            print("✅ 이미지 토큰 수가 기존 결과와 같음 (같은 해상도·프로세서)")
        if _agree < 6: print("⚠️ 답이 기존과 많이 다릅니다. 계속 진행하지만 결과를 확인하세요")
print("✅ 사전 점검 통과 → 본 채점을 시작합니다")

## 10. 검증 1,000문항 채점

In [ ]:
import base64, io, html as _html
def export_error_report(m, path, max_items=150):
    wrong = m[~m.correct].head(max_items); parts = [f"<h2>{_html.escape(os.path.basename(path))} — 오답 {int((~m.correct).sum())}건 / {len(m)}문항</h2>"]
    for _, r in wrong.iterrows():
        im = load_image(r.path); im.thumbnail((560, 560)); buf = io.BytesIO(); im.save(buf, "JPEG", quality=72)
        opts = "".join(f"<li>({k}) {_html.escape(str(r[k]))} — p={r['p_' + k]:.2f}{' <b>← 정답</b>' if k == r.answer else ''}{' <b style=color:#c00>← 예측</b>' if k == r.pred else ''}</li>" for k in LETTERS)
        parts.append(f"<div style='display:flex;gap:16px;margin:18px 0;border-top:1px solid #ccc;padding-top:12px'><img src='data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}'>"
                     f"<div><b>{r.id}</b><p>{_html.escape(str(r.question))}</p><ul>{opts}</ul></div></div>")
    open(path, "w", encoding="utf-8").write("<meta charset='utf-8'><body style='font-family:sans-serif;max-width:1100px;margin:auto'>" + "".join(parts))
    print("오답 리포트 저장:", path)

FINAL_TAG = f"final_ep{SRC_EP}"      # stage2와 같은 이름 규칙 → preds_final_ep{N}__{PRESET}__1000.csv / preds_final_ep{N}_test__{PRESET}__6714.csv
set_resolution(PRESET)
if DO_VAL:
    acc_new, m_new = evaluate(val_df, FINAL_TAG)          # 끊겨도 다시 실행하면 남은 문항만 채점
    export_error_report(m_new, os.path.join(RUN_DIR, f"error_report_{FINAL_TAG}_{PRESET}.html"))
else:
    print("검증셋까지 학습한 실행이라 검증 채점을 건너뜁니다 (test만 채점)")

## 11. 기존 NF4 결과와 문항 단위 비교 → 판정
- **채택**: 순이득 ≥ +3 이고 NLL 감소 (보고서 E1 기준) → test 채점으로 넘어감
- **보류**: 순이득은 +1 이상이지만 기준 미달(+1~+2, 또는 NLL 증가) → test는 자동으로 돌리지 않음. 단, 현재 최고 조합에서 이 멤버만 바꿨을 때 순이득이 +3 이상이면(조합 재현이 확인된 경우) test를 채점
- **불채택**: 순이득 0 이하
- **확인 필요**: letter_mass가 0.01 넘게 떨어졌거나 이미지 토큰 수가 기존과 다름 → 비교가 성립하지 않으므로 판정하지 않음

In [ ]:
def _probs(d): return d[["p_a", "p_b", "p_c", "p_d"]].to_numpy(dtype=float)
def _load_preds(path, ids):
    d = pd.read_csv(path).drop_duplicates("id", keep="last").set_index("id")
    miss = [i for i in ids if i not in d.index]
    assert not miss, f"{os.path.basename(path)}에 없는 문항 {len(miss)}개 (예: {miss[:3]})"
    return d.loc[ids]
def _nll(P, y): return float(-np.log(np.clip(P[np.arange(len(y)), y], 1e-12, 1.0)).mean())
def _paired(a_old, a_new, y):
    c0, c1 = a_old == y, a_new == y
    fixed, broken = int((~c0 & c1).sum()), int((c0 & ~c1).sum())
    return fixed, broken, fixed - broken, int((a_old != a_new).sum())
def find_val_file(run, preset):
    d = os.path.join(OUT_DIR, run)
    if not os.path.isdir(d): return None, "폴더 없음"
    rx = re.compile(rf"^preds_final_ep(\d+)__{re.escape(preset)}__1000\.csv$")
    c = {int(m.group(1)): f for f in os.listdir(d) if (m := rx.match(f))}
    if not c: return None, "검증 파일 없음"
    if len(c) == 1: return os.path.join(d, next(iter(c.values()))), ""
    lp = os.path.join(d, "train_log.json")
    ea = {int(k): v for k, v in (json.load(open(lp, encoding="utf-8")).get("epoch_acc") or {}).items()} if os.path.exists(lp) else {}
    ep = max(ea, key=lambda e: (ea[e], e)) if ea else None
    ep = ep if ep in c else max(c)
    return os.path.join(d, c[ep]), f"ep{ep} 사용(여러 개 중)"

V = {"verdict": "판정 불가", "net": None, "ens_net": None, "lines": []}
def say(s): print(s); V["lines"].append(s)
AUTO_TEST_OK = False
if not DO_VAL:
    V["verdict"] = "검증 생략 (검증셋까지 학습한 실행) → test만 채점"; AUTO_TEST_OK = True
else:
    ids = val_df.id.tolist(); y = val_df.answer.map(LETTERS.index).to_numpy()
    new = _load_preds(os.path.join(RUN_DIR, f"preds_{FINAL_TAG}__{PRESET}__1000.csv"), ids); P1 = _probs(new); a1 = P1.argmax(1)
    if REF_VAL is None:
        say(f"bf16 검증 정확도 {(a1 == y).mean() * 100:.2f}% — 기존 검증 파일이 없어 문항 단위 비교는 못 합니다")
        V["verdict"] = f"판정 불가 — {SRC_RUN}에 preds_final_ep{SRC_EP}__{PRESET}__1000.csv가 없어 NF4와 비교할 수 없습니다"
    else:
        ref = _load_preds(REF_VAL, ids); P0 = _probs(ref); a0 = P0.argmax(1)
        fixed, broken, net, D = _paired(a0, a1, y)
        nll0, nll1 = _nll(P0, y), _nll(P1, y)
        lm0, lm1 = float(ref.letter_mass.mean()), float(new.letter_mass.mean())
        dp = np.abs(P1 - P0).max(1)
        say(f"[단독] {SRC_RUN}@{PRESET}, 같은 1,000문항 |  기존(NF4) → bf16")
        say(f"  정확도 {(a0 == y).mean() * 100:.2f}% → {(a1 == y).mean() * 100:.2f}% | NLL {nll0:.4f} → {nll1:.4f} | letter_mass {lm0:.3f} → {lm1:.3f}")
        say(f"  고침 {fixed} | 새로 틀림 {broken} | 순이득 {net:+d} | 답이 바뀐 문항 D={D} (강한 근거 기준 2√D = {2 * math.sqrt(D):.1f})")
        say(f"  문항별 확률 변화(최대 |Δp|): 중앙값 {np.median(dp):.4f}, 상위 10% {np.quantile(dp, 0.9):.4f}")
        health = []                                                  # 비교 자체가 성립하는지 (하나라도 걸리면 판정하지 않음)
        if "img_tokens" in ref.columns:
            _same = int((ref.img_tokens.to_numpy() == new.img_tokens.to_numpy()).sum())
            say(f"  이미지 토큰 수 일치 {_same}/{len(ids)}")
            if _same != len(ids): health.append(f"이미지 토큰 수가 다른 문항 {len(ids) - _same}개(해상도 불일치)")
        if lm1 < lm0 - 0.01: health.append(f"letter_mass 하락 {lm0:.3f} → {lm1:.3f}(토큰화·템플릿 문제 가능성)")
        if D <= 2 and np.median(dp) < 0.002: say("  ℹ️ 기존 결과와 사실상 같습니다 → 기존 실행도 bf16이었을 가능성이 큽니다")
        _chg = val_df.assign(NF4=[LETTERS[i] for i in a0], bf16=[LETTERS[i] for i in a1],
                             p_gold_NF4=P0[np.arange(len(y)), y].round(3), p_gold_bf16=P1[np.arange(len(y)), y].round(3))
        _chg = _chg[_chg.NF4 != _chg.bf16].copy()
        _chg["결과"] = np.where(_chg.bf16 == _chg.answer, "고침", np.where(_chg.NF4 == _chg.answer, "새로 틀림", "둘 다 틀림"))
        _cols = ["id", "결과", "answer", "NF4", "bf16", "p_gold_NF4", "p_gold_bf16", "question", "a", "b", "c", "d"]
        _chg[_cols].to_csv(os.path.join(RUN_DIR, f"e1_changed_{PRESET}.csv"), index=False, encoding="utf-8-sig")
        if len(_chg):
            print("\n답이 바뀐 문항 (e1_changed_" + PRESET + ".csv에 전체 저장):")
            print(_chg[["id", "결과", "answer", "NF4", "bf16", "p_gold_NF4", "p_gold_bf16", "question"]].head(20).to_string(index=False))
        V.update({"acc_nf4": round(float((a0 == y).mean() * 100), 2), "acc_bf16": round(float((a1 == y).mean() * 100), 2),
                  "fixed": fixed, "broken": broken, "net": net, "D": D, "nll_nf4": round(nll0, 4), "nll_bf16": round(nll1, 4)})
        if health:
            V["verdict"] = f"확인 필요 — {'; '.join(health)} → 판정하지 않습니다. 위 출력을 확인하세요"
        elif net >= NET_MIN and nll1 < nll0:
            V["verdict"] = f"채택 — 순이득 {net:+d} ≥ +{NET_MIN}, NLL 감소 (보고서 E1 기준 통과)" + \
                           ("" if net >= 2 * math.sqrt(D) else " / 단, 2√D 미만이라 공개 제출로 한 번 더 확인 권장")
            AUTO_TEST_OK = True
        elif net >= 1:
            V["verdict"] = f"보류 — 순이득 {net:+d}" + (" (NLL은 늘어남)" if nll1 >= nll0 else "") + " → 기준 미달. 공개 제출로 확인하려면 test를 직접 켜세요"
            V["hold"] = True
        else:
            V["verdict"] = f"불채택 — 순이득 {net:+d}: bf16 재채점 이득 없음"

    # ── 앙상블 관점 (참고): 기준 조합에서 SRC_RUN 자리만 bf16 결과로 교체 ──
    if ENSEMBLE_REF:
        mem, w = [tuple(x) for x in ENSEMBLE_REF["members"]], list(ENSEMBLE_REF["weights"])
        assert len(mem) == len(w), "ENSEMBLE_REF의 members와 weights 개수가 다릅니다"
        found = [find_val_file(r, p) for r, p in mem]
        miss = [f"{r}@{p}({why})" for (r, p), (f, why) in zip(mem, found) if f is None]
        if miss:
            say(f"[앙상블] {ENSEMBLE_REF['name']}: 멤버 파일이 없어 건너뜁니다 → {miss}")
        elif (SRC_RUN, PRESET) not in mem:
            say(f"[앙상블] {ENSEMBLE_REF['name']}에 {SRC_RUN}@{PRESET}가 없어 교체 비교를 건너뜁니다")
        else:
            Ps = [_probs(_load_preds(f, ids)) for f, _ in found]
            ens = lambda Ps_: sum(wi * Pi for wi, Pi in zip(w, Ps_)) / sum(w)
            e0 = ens(Ps).argmax(1); acc_e0 = (e0 == y).mean() * 100
            exp = ENSEMBLE_REF.get("expected_val_acc")
            ok_rep = exp is None or abs(acc_e0 - exp) < 0.05
            Ps2 = list(Ps); Ps2[mem.index((SRC_RUN, PRESET))] = P1; e1 = ens(Ps2).argmax(1)
            f2, b2, n2, D2 = _paired(e0, e1, y)
            say(f"[앙상블] {ENSEMBLE_REF['name']} 재현 {acc_e0:.2f}%" + (f" (기대 {exp})" if exp is not None else "") +
                ("" if ok_rep else "  ⚠️ 재현이 안 됩니다 → Stage 3이 쓴 멤버 파일과 다를 수 있어 아래 줄은 참고만 하세요"))
            say(f"  {SRC_RUN} 자리만 bf16으로 교체 → {(e1 == y).mean() * 100:.2f}% | 고침 {f2} | 새로 틀림 {b2} | 순이득 {n2:+d} | 답이 바뀐 문항 {D2}")
            _used = [f"{r}:{os.path.basename(f)}{('(' + why + ')') if why else ''}" for (r, _), (f, why) in zip(mem, found)]
            print("  사용한 멤버 파일:", _used)
            V["ens_net"] = n2
            if V.get("hold") and ok_rep and n2 >= NET_MIN:           # 단독이 +1~+2(음수 아님)이고 앙상블에서 +3 이상일 때만
                AUTO_TEST_OK = True; V["verdict"] += f" / 앙상블 교체 순이득 {n2:+d} ≥ +{NET_MIN} → test 채점"
say("판정: " + V["verdict"])
E1_INFO.update({f"result_{PRESET}": {k: v for k, v in V.items() if k != "lines"}})
json.dump(E1_INFO, open(INFO_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)

## 12. test 추론 → 제출 파일 (판정 통과 시)

In [ ]:
# RUN_TEST = True     # ← 판정과 상관없이 test를 돌리려면 이 줄 맨 앞의 #을 지우고 이 셀만 다시 실행 (1~1.5시간)
DO_TEST = (RUN_TEST is True) or (RUN_TEST == "auto" and AUTO_TEST_OK)
if not DO_TEST:
    print(f"test 추론 생략 (RUN_TEST={RUN_TEST!r} | 판정: {V['verdict']})")
else:
    set_resolution(PRESET)
    pred, TEST_PATH = predict_df(test_df, f"{FINAL_TAG}_test")        # 끊겨도 1번 셀부터 다시 실행하면 남은 문항만 채점
    sub = sub_df[["id"]].merge(pred[["id", "pred"]], on="id", how="left").rename(columns={"pred": "answer"})
    assert len(sub) == len(sub_df) and list(sub.id) == list(sub_df.id), "id 순서/개수 불일치"
    assert sub.answer.notna().all(), f"예측 누락 {int(sub.answer.isna().sum())}건"
    assert set(sub.answer.unique()) <= set(LETTERS), sub.answer.unique()
    sp = os.path.join(RUN_DIR, f"submission_{RUN_NAME}_{FINAL_TAG}_{PRESET}.csv"); sub.to_csv(sp, index=False)
    print("저장:", TEST_PATH, "\n저장:", sp, "| 예측 분포:", sub.answer.value_counts().sort_index().to_dict())
    V["test_file"] = TEST_PATH
    if REF_TEST:
        _t0 = pd.read_csv(REF_TEST).drop_duplicates("id", keep="last").set_index("id").reindex(sub.id)
        if _t0.pred.notna().all():
            V["test_diff"] = int((_t0.pred.to_numpy() != sub.answer.to_numpy()).sum())
            print(f"기존(NF4) test 예측과 답이 다른 문항: {V['test_diff']}개 / {len(sub)} (공개 절반에는 대략 그 절반)")
    E1_INFO.update({f"test_{PRESET}": {"file": TEST_PATH, "diff_vs_nf4": V.get("test_diff")}})
    json.dump(E1_INFO, open(INFO_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)

## 13. 결과 요약

In [ ]:
print(f"E1 | {SRC_RUN} → {RUN_NAME} | {MODEL_ID} bf16 + adapter_ep{SRC_EP} (md5 {ADAPTER_MD5[:8]}) | {PRESET} | {GPU_NAME}")
for _l in V["lines"]: print(_l)
if V.get("test_file"): print("test 예측:", os.path.basename(V["test_file"]), "| 기존 NF4와 다른 답:", V.get("test_diff", "비교 파일 없음"))
if torch.cuda.is_available(): print(f"최대 VRAM 사용: {torch.cuda.max_memory_allocated() / 2**30:.1f}GB / {VRAM_GB:.0f}GB")
if V.get("test_file"): print(f'\nStage 3에서 쓰려면 멤버 ("{SRC_RUN}", "{PRESET}") 대신 ("{RUN_NAME}", "{PRESET}")')
if V["verdict"].startswith("채택"):
    print("다음: 같은 노트북으로 Q38·Gemma 어댑터도 재채점 (1번 셀 SRC_RUN·MODEL_ID만 변경) → Stage 3에서 bf16 멤버로 바꾼 조합 비교")
elif V["verdict"].startswith(("보류", "불채택")):
    print("다음: 위 출력을 확인하세요 (bf16 재학습을 할지 판단)")